# Πρόβλεψη Μηνιαίων Αξιώσεων Ασφάλισης Αυτοκινήτου με PROC FORECAST

## Σύνοψη για τη Διοίκηση

Μια ομάδα αναλογιστικών αποθεμάτων χρειάζεται μια εικόνα 12 μηνών μπροστά για τις μηνιαίες αξιώσεις ασφάλισης αυτοκινήτου, οι οποίες δείχνουν σταθερή ανάπτυξη χαρτοφυλακίου συν μια έντονη χειμερινή άνοδο. Αυτό το notebook παράγει πέντε έτη συνθετικών μηνιαίων αξιώσεων (60 μήνες, Ιαν 2020 - Δεκ 2024, με εύρος περίπου 1.460 έως 2.780 αξιώσεις), και στη συνέχεια χρησιμοποιεί την **PROC FORECAST** για να προσαρμόσει μια βάση σταδιακής αυτοπαλινδρόμησης (stepwise-autoregressive) και ένα πολλαπλασιαστικό εποχιακό μοντέλο Holt-Winters. Κάθε μοντέλο παράγει 12 μηνιαίες σημειακές προβλέψεις με όρια εμπιστοσύνης 95% για τον σχεδιασμό χωρητικότητας και αποθεμάτων. Το εποχιακό μοντέλο Holt-Winters προβλέπει την ισχυρότερη ζήτηση έναν έως δύο μήνες μπροστά (περίπου 2.780-2.900 αξιώσεις), η οποία υποχωρεί σε ένα χαμηλό γύρω στο βήμα εννέα (περίπου 2.200), ενώ η βάση αυτοπαλινδρόμησης προβλέπει μια πιο ομαλή φθίνουσα πορεία· και οι δύο ζώνες εμπιστοσύνης διευρύνονται με τον ορίζοντα.

## Πηγές Δεδομένων

| Σύνολο Δεδομένων | Γραμμές | Κλίμακα | Βασικές Μεταβλητές | Περιγραφή |
|---------|------|-------|---------------|-------------|
| `claims` | 60 | Μία γραμμή ανά ημερολογιακό μήνα (Ιαν 2020 - Δεκ 2024) | `date` (ημερομηνία SAS, `MONYY7.`), `claim_count` (αριθμητική) | Συνθετικές μηνιαίες αξιώσεις ασφάλισης αυτοκινήτου, φτιαγμένες από μια γραμμική τάση ανάπτυξης (~9 αξιώσεις/μήνα), έναν ημιτονοειδή ετήσιο κύκλο, μια προσθετική χειμερινή άνοδο Δεκ/Ιαν/Φεβ και τυχαίο θόρυβο Gauss (`rand('normal')`). Ο σπόρος `20240531` την καθιστά αναπαραγώγιμη. |

# Πρόβλεψη Μηνιαίων Αξιώσεων Ασφάλισης Αυτοκινήτου

Οι ομάδες αποθεμάτων και σχεδιασμού χωρητικότητας μιας ασφαλιστικής προσωπικών κλάδων χρειάζονται μια μπροστινή εικόνα του πόσες **αξιώσεις αυτοκινήτου** θα αναφέρονται κάθε μήνα. Ο όγκος αξιώσεων σε αυτό το χαρτοφυλάκιο αυξάνεται σταθερά καθώς επεκτείνεται το χαρτοφυλάκιο, και αιχμηρώνει κάθε χειμώνα όταν ο πάγος, το χιόνι και το μειωμένο φως της ημέρας αυξάνουν τις αξιώσεις σύγκρουσης και υαλοπινάκων.

Αυτό το notebook διατρέχει μια πλήρη ροή εργασίας `PROC FORECAST`:

1. Παραγωγή μιας ρεαλιστικής, πλήρως συνθετικής σειράς μηνιαίου πλήθους αξιώσεων.
2. Προσαρμογή μιας βάσης **σταδιακής αυτοπαλινδρόμησης (STEPAR)** που αποτυπώνει την τάση συν την αυτοσυσχέτιση.
3. Προσαρμογή ενός **πολλαπλασιαστικού μοντέλου Holt-Winters (WINTERS)** που μοντελοποιεί ρητά τον 12μηνο εποχιακό κύκλο.
4. Σύγκριση των δύο μοντέλων και ερμηνεία της μπροστινής πρόβλεψης και της ζώνης εμπιστοσύνης.

Όλα εκτελούνται σε ενσωματωμένα συνθετικά δεδομένα — χωρίς εξωτερικά αρχεία ή πρόσβαση σε δίκτυο.

## Βήμα 1 — Παραγωγή της συνθετικής σειράς αξιώσεων

Το παρακάτω βήμα DATA δημιουργεί **60 μηνιαίες παρατηρήσεις** (Ιαν 2020 έως Δεκ 2024). Για κάθε μήνα συνδυάζουμε τέσσερις συνιστώσες που αντικατοπτρίζουν ένα πραγματικό χαρτοφυλάκιο αυτοκινήτου:

- **Τάση** — μια βάση ~1.800 αξιώσεων που αυξάνεται κατά ~9 ανά μήνα καθώς αυξάνεται η έκθεση.
- **Ετήσιος κύκλος** — ένας όρος ημιτόνου/συνημιτόνου που δίνει ένα ομαλό εποχιακό κύμα.
- **Χειμερινή άνοδος** — μια προσθετική αύξηση τον Δεκέμβριο, Ιανουάριο και Φεβρουάριο.
- **Θόρυβος** — `rand('normal', 0, 90)` για ρεαλιστική διακύμανση από μήνα σε μήνα.

Η `call streaminit()` σταθεροποιεί τη ροή τυχαίων αριθμών ώστε το notebook να είναι αναπαραγώγιμο. Η μεταβλητή `date` είναι μια πραγματική ημερομηνία SAS φτιαγμένη με `INTNX` και μορφοποιημένη `MONYY7.`, και η δήλωση `ID date INTERVAL=MONTH` την ονομάζει ως τον χρονικό αναγνωριστή. Οι πρώτες 14 γραμμές που εκτυπώνονται παρακάτω κυμαίνονται μεταξύ περίπου 1.460 και 2.450 αξιώσεων.

In [1]:
ΔΕΔΟΜΕΝΑ claims;
    CALL streaminit(20240531);
    ΕΠΑΝΑΛΗΨΗ i = 0 ΕΩΣ 59;
        /* Δημιουργία πραγματικής ημερομηνίας SAS ώστε το INTERVAL=MONTH να ευθυγραμμίζεται */
        date = intnx('month', '01JAN2020'd, i);
        ΜΟΡΦΗ date monyy7.;

        month_idx = mod(i, 12) + 1;          /* 1 = Ιαν ... 12 = Δεκ */

        trend  = 1800 + 9 * i;               /* αυξανόμενη βάση έκθεσης   */
        season = 260 * sin((month_idx - 1) / 12 * 2 * constant('pi'))
               + 150 * cos((month_idx - 1) / 12 * 2 * constant('pi'));
        winter = 220 * (month_idx IN (12, 1, 2));   /* αύξηση λόγω πάγου/χιονιού  */
        noise  = rand('normal', 0, 90);

        claim_count = round(trend + season + winter + noise);
        ΕΑΝ claim_count < 0 ΤΟΤΕ claim_count = 0;
        ΕΞΟΔΟΣ;
    ΤΕΛΟΣ;
    ΚΡΑΤΗΣΗ date claim_count;
ΕΚΤΕΛΕΣΗ;

ΔΙΑΔΙΚΑΣΙΑ ΕΚΤΥΠΩΣΗ ΔΕΔΟΜΕΝΑ=claims(obs=14) ΕΤΙΚΕΤΑ;
    ΕΤΙΚΕΤΑ date = 'Μήνας' claim_count = 'Αναφερθείσες Αξιώσεις';
    TITLE 'Πρώτοι 14 Μήνες Συνθετικών Αξιώσεων Αυτοκινήτου';
ΕΚΤΕΛΕΣΗ;

                                    Πρώτοι 14 Μήνες Συνθετικών Αξιώσεων Αυτοκινήτου                                     

  Obs       Μήνας                      Αναφερθείσες Αξιώσεις
    1       21915                                       2178
    2       21946                                       2281
    3       21975                                       2252
    4       22006                                       1974
    5       22036                                       2067
    6       22067                                       1805
    7       22097                                       1697
    8       22128                                       1619
    9       22159                                       1462
   10       22189                                       1688
   11       22220                                       1713
   12       22250                                       2008
   13       22281                                       2116
   14       22312       


NOTE: DATA claims


NOTE: Wrote claims (60 rows, 2 columns).
NOTE: DATA elapsed:
  wall  0.01 seconds
  cpu   0.01 seconds
NOTE: PROC PRINT data=claims

NOTE: PROC PRINT completed: 14 observations printed, 2 variables


## Βήμα 2 — Βάση σταδιακής αυτοπαλινδρόμησης (METHOD=STEPAR)

Η `METHOD=STEPAR` είναι η προεπιλογή. Πρώτα προσαρμόζει ένα μοντέλο χρονικής τάσης — εδώ `TREND=2` για γραμμική τάση — και στη συνέχεια εφαρμόζει μια **σταδιακή αυτοπαλινδρόμηση** στα κατάλοιπα, εισάγοντας και διατηρώντας υστερήσεις AR βάσει σημαντικότητας. Το `AR=4` περιορίζει την υποψήφια τάξη αυτοπαλινδρόμησης σε τέσσερις υστερήσεις, αρκετό για μια μηνιαία σειρά με βραχυπρόθεσμη ορμή.

Βασικές επιλογές που χρησιμοποιούνται εδώ:

- `LEAD=12` — πρόβλεψη 12 μηνών πέρα από τα δεδομένα.
- `ALPHA=0.05` — όρια εμπιστοσύνης 95%.
- `OUTFULL` — στοίβαξη των ιστορικών γραμμών `ACTUAL` μαζί με τις γραμμές του ορίζοντα πρόβλεψης στο σύνολο δεδομένων `OUT=` (αντί μόνο των γραμμών πρόβλεψης).
- `OUTLIMIT` — προσθήκη των **στηλών** κάτω/άνω ορίου εμπιστοσύνης (`L95_claim_count`, `U95_claim_count`).
- `ID date INTERVAL=MONTH` — ονομάζει τον χρονικό αναγνωριστή και δηλώνει ότι η σειρά είναι μηνιαία.

In [2]:
ΔΙΑΔΙΚΑΣΙΑ forecast ΔΕΔΟΜΕΝΑ=claims
        out=fc_stepar
        METHOD=stepar trend=2 ar=4
        LEAD=12 ALPHA=0.05
        outfull outlimit;
    id date interval=month;
    ΜΕΤΑΒΛΗΤΗ claim_count;
ΕΚΤΕΛΕΣΗ;

ΔΙΑΔΙΚΑΣΙΑ ΕΚΤΥΠΩΣΗ ΔΕΔΟΜΕΝΑ=fc_stepar(obs=10) ΕΤΙΚΕΤΑ;
    ΕΤΙΚΕΤΑ date = 'Ημερομηνία' _type_ = 'Τύπος' claim_count = 'Αξιώσεις'
          l95_claim_count = 'Κάτω 95%' u95_claim_count = 'Άνω 95%';
    TITLE 'Έξοδος STEPAR: Γραμμές Πραγματικών, Προσαρμοσμένων και Πρόβλεψης';
ΕΚΤΕΛΕΣΗ;

                                    Πρώτοι 14 Μήνες Συνθετικών Αξιώσεων Αυτοκινήτου                                     

The FORECAST Procedure

Variables Forecasted: claim_count
Number of Observations: 60
Forecast Periods: 12



                            Έξοδος STEPAR: Γραμμές Πραγματικών, Προσαρμοσμένων και Πρόβλεψης                            

  Obs            Ημερομηνία       Τύπος          Αξιώσεις      Κάτω 95%     Άνω 95%
    1                 21915  ACTUAL           2121.816667             .           .
    2                 21946  ACTUAL           2167.711419             .           .
    3                 21975  ACTUAL           2254.781177             .           .
    4                 22006  ACTUAL           2228.505912             .           .
    5                 22036  ACTUAL           1978.152296             .           .
    6                 22067  ACTUAL           2030.606563             .           .
    7                 22097  ACTUAL           1864.520418  


NOTE: PROC FORECAST data=claims

NOTE: Using Python for FORECAST estimation
NOTE: Output dataset created with 72 observations.
NOTE: PROC PRINT data=fc_stepar

NOTE: PROC PRINT completed: 10 observations printed, 5 variables


### Ανάγνωση του συνόλου δεδομένων OUT=

Το σύνολο δεδομένων `OUT=` στοιβάζει γραμμές ανά στήλη `_TYPE_` και φέρει τα όρια εμπιστοσύνης ως πλευρικές **στήλες**:

| Στοιχείο | Είδος | Σημασία |
|---------|------|---------|
| `_TYPE_ = 'ACTUAL'` | γραμμή | Το παρατηρηθέν `claim_count` για κάθε έναν από τους 60 ιστορικούς μήνες. |
| `_TYPE_ = 'FORECAST'` | γραμμή | Οι 12 σημειακές προβλέψεις για τον ορίζοντα πρόβλεψης. |
| `L95_claim_count` / `U95_claim_count` | στήλη | Κάτω / άνω όρια εμπιστοσύνης 95%, συμπληρωμένα στις γραμμές `FORECAST` (ελλείπουν στις γραμμές `ACTUAL`). Το αριθμητικό επίπεδο αντικατοπτρίζει το `ALPHA=`. |

Έτσι το εκτυπωμένο `OUT=` εδώ περιέχει 72 γραμμές: 60 ιστορικές γραμμές `ACTUAL` ακολουθούμενες από 12 γραμμές ορίζοντα `FORECAST`. Οι πρώτες δέκα γραμμές που εμφανίζονται παραπάνω είναι όλες `ACTUAL`, με τις στήλες ορίων εμπιστοσύνης να λείπουν επειδή τα όρια επισυνάπτονται μόνο στις γραμμές πρόβλεψης.

> **Σημείωση μηχανής.** Το SAS `OUTFULL` γράφει επίσης μια εντός-δείγματος γραμμή πρόβλεψης ενός βήματος `FORECAST` για κάθε ιστορικό μήνα, και το `OUTRESID` προσθέτει γραμμές `_TYPE_='RESIDUAL'`. Η PROC FORECAST του Jenner δεν εκπέμπει ακόμη αυτές τις εντός-δείγματος προσαρμοσμένες/υπολειμματικές γραμμές (καταγεγραμμένο ως δοκιμή κενού `400979`), οπότε αυτό το notebook διαβάζει μόνο το ιστορικό `ACTUAL` και τον μπροστινό ορίζοντα `FORECAST`.

## Βήμα 3 — Εποχιακό μοντέλο Holt-Winters (METHOD=WINTERS)

Το μοντέλο STEPAR αποτυπώνει την τάση και τη βραχυπρόθεσμη αυτοσυσχέτιση, αλλά δεν μοντελοποιεί ρητά την επαναλαμβανόμενη άνοδο Δεκεμβρίου–Φεβρουαρίου. Για μια σειρά με σαφή ετήσιο ρυθμό, το **πολλαπλασιαστικό Holt-Winters** είναι το καλύτερο εργαλείο.

Η `METHOD=WINTERS` αποσυνθέτει τη σειρά σε επίπεδο, γραμμική τάση (`TREND=2`) και έναν **πολλαπλασιαστικό εποχιακό συντελεστή**. Το `SEASONS=12` δηλώνει έναν εποχιακό κύκλο 12 περιόδων (μηνιαίο). Ζητάμε ξανά ένα `LEAD` 12 μηνών με όρια 95% και `OUTFULL OUTLIMIT` ώστε η έξοδος να ευθυγραμμίζεται με την εκτέλεση STEPAR.

Επειδή η μηχανή προάγει το `ID` πρόβλεψης κατά μία μονάδα ανά βήμα (αντί να τηρεί το `INTERVAL=MONTH` για τις ημερομηνίες του ορίζοντα — δοκιμή κενού `400979`), το παρακάτω κελί εξετάζει την πρόβλεψη ανά **μήνες μπροστά (βήμα 1-12)** αντί να βασίζεται στις εκτυπωμένες ημερολογιακές ημερομηνίες.

In [3]:
ΔΙΑΔΙΚΑΣΙΑ forecast ΔΕΔΟΜΕΝΑ=claims
        out=fc_winters
        METHOD=winters seasons=12 trend=2
        LEAD=12 ALPHA=0.05
        outfull outlimit;
    id date interval=month;
    ΜΕΤΑΒΛΗΤΗ claim_count;
ΕΚΤΕΛΕΣΗ;

/* Διατήρηση του 12μηνου μπροστινού ορίζοντα και ευρετηρίασή του ανά βήμα (1..12). */
ΔΕΔΟΜΕΝΑ horizon;
    ΟΡΙΣΜΟΣ fc_winters;
    RETAIN months_ahead 0;
    ΕΑΝ _type_ = 'FORECAST';
    months_ahead + 1;
    ΚΡΑΤΗΣΗ months_ahead claim_count l95_claim_count u95_claim_count;
ΕΚΤΕΛΕΣΗ;

ΔΙΑΔΙΚΑΣΙΑ ΕΚΤΥΠΩΣΗ ΔΕΔΟΜΕΝΑ=horizon ΕΤΙΚΕΤΑ;
    ΕΤΙΚΕΤΑ months_ahead     = 'Μήνες Μπροστά'
          claim_count       = 'Προβλεπόμενες Αξιώσεις'
          l95_claim_count   = 'Κάτω 95%'
          u95_claim_count   = 'Άνω 95%';
    TITLE 'Πρόβλεψη Holt-Winters 12 Μηνών (ανά βήμα)';
ΕΚΤΕΛΕΣΗ;

                            Έξοδος STEPAR: Γραμμές Πραγματικών, Προσαρμοσμένων και Πρόβλεψης                            

The FORECAST Procedure

Variables Forecasted: claim_count
Number of Observations: 60
Forecast Periods: 12



                                       Πρόβλεψη Holt-Winters 12 Μηνών (ανά βήμα)                                        

  Obs              Μήνες Μπροστά                       Προβλεπόμενες Αξιώσεις      Κάτω 95%      Άνω 95%
    1                          1                                   2783.07951   2577.844742  2988.314278
    2                          2                                  2897.355557   2607.109764  3187.601349
    3                          3                                  2805.272075   2449.795029   3160.74912
    4                          4                                  2664.498049   2254.028514  3074.967585
    5                          5                                  2628.810136   2169.891244  3087.729029
    6            


NOTE: PROC FORECAST data=claims

NOTE: Using Python for FORECAST estimation
NOTE: Output dataset created with 72 observations.
NOTE: DATA horizon


NOTE: Read 72 rows from fc_winters.
NOTE: Wrote horizon (12 rows, 4 columns).
NOTE: DATA elapsed:
  wall  0.00 seconds
  cpu   0.00 seconds
NOTE: PROC PRINT data=horizon

NOTE: PROC PRINT completed: 12 observations printed, 4 variables


## Βήμα 4 — Σύγκριση των δύο μοντέλων απευθείας

Ο σαφέστερος τρόπος να κρίνουμε αν το εποχιακό μοντέλο αξίζει τον κόπο είναι να τοποθετήσουμε την 12μηνη πρόβλεψή του δίπλα στη βάση STEPAR, βήμα προς βήμα. Αντλούμε τις γραμμές `FORECAST` και από τα δύο σύνολα δεδομένων `OUT=`, τις ευρετηριάζουμε ανά μήνες-μπροστά και τις συγχωνεύουμε ώστε η απόκλιση να είναι ορατή με μια ματιά.

Οι δύο μέθοδοι συμφωνούν στο γενικό επίπεδο αλλά διαφωνούν στο **σχήμα**: το Holt-Winters μεταφέρει τον ετήσιο ρυθμό στον ορίζοντα (ένα υψηλότερο επίπεδο στον πρώιμο ορίζοντα που υποχωρεί σε ένα ενδιάμεσο χαμηλό και ανεβαίνει ξανά), ενώ το STEPAR — που μοντελοποιεί την εποχικότητα μόνο έμμεσα μέσω υστερήσεων AR — φθίνει πιο ομαλά προς τον μέσο όρο της σειράς. Το κενό μεταξύ τους σε κάθε βήμα είναι το εποχιακό σήμα που το STEPAR αφήνει στο τραπέζι.

> Το SAS θα οδηγούσε κανονικά αυτόν τον έλεγχο επάρκειας με κατάλοιπα ενός βήματος `OUTRESID` (`_TYPE_='RESIDUAL'`). Το Jenner δεν συμπληρώνει ακόμη αυτές τις γραμμές (δοκιμή κενού `400979`), οπότε συγκρίνουμε απευθείας τις δύο μπροστινές προβλέψεις — ένα διαγνωστικό που χρησιμοποιεί μόνο έξοδο που πράγματι παράγει η μηχανή.

In [4]:
/* Ορίζοντας STEPAR, ευρετηριασμένος ανά μήνες-μπροστά. */
ΔΕΔΟΜΕΝΑ stepar_h;
    ΟΡΙΣΜΟΣ fc_stepar;
    RETAIN months_ahead 0;
    ΕΑΝ _type_ = 'FORECAST';
    months_ahead + 1;
    stepar = claim_count;
    ΚΡΑΤΗΣΗ months_ahead stepar;
ΕΚΤΕΛΕΣΗ;

/* Ορίζοντας WINTERS, ευρετηριασμένος ανά μήνες-μπροστά. */
ΔΕΔΟΜΕΝΑ winters_h;
    ΟΡΙΣΜΟΣ fc_winters;
    RETAIN months_ahead 0;
    ΕΑΝ _type_ = 'FORECAST';
    months_ahead + 1;
    winters = claim_count;
    ΚΡΑΤΗΣΗ months_ahead winters;
ΕΚΤΕΛΕΣΗ;

/* Συγχώνευση των δύο και εμφάνιση του εποχιακού κενού που το STEPAR χάνει. */
ΔΕΔΟΜΕΝΑ COMPARE;
    ΣΥΓΧΩΝΕΥΣΗ stepar_h winters_h;
    ΚΑΤΑ months_ahead;
    seasonal_gap = winters - stepar;
ΕΚΤΕΛΕΣΗ;

ΔΙΑΔΙΚΑΣΙΑ ΕΚΤΥΠΩΣΗ ΔΕΔΟΜΕΝΑ=COMPARE ΕΤΙΚΕΤΑ;
    ΕΤΙΚΕΤΑ months_ahead = 'Μήνες Μπροστά'
          stepar        = 'Πρόβλεψη STEPAR'
          winters       = 'Πρόβλεψη Winters'
          seasonal_gap  = 'Winters - STEPAR';
    ΜΟΡΦΗ stepar winters seasonal_gap 8.0;
    TITLE 'Σύγκριση STEPAR έναντι Holt-Winters: Πρόβλεψη 12 Μηνών';
ΕΚΤΕΛΕΣΗ;

                                 Σύγκριση STEPAR έναντι Holt-Winters: Πρόβλεψη 12 Μηνών                                 

  Obs              Μήνες Μπροστά          Πρόβλεψη STEPAR          Πρόβλεψη Winters  Winters - STEPAR
    1                          1                     2619                      2783               164
    2                          2                     2537                      2897               361
    3                          3                     2384                      2805               421
    4                          4                     2214                      2664               450
    5                          5                     2089                      2629               540
    6                          6                     2010                      2548               539
    7                          7                     1982                      2392               410
    8                          8                     1995     


NOTE: DATA stepar_h


NOTE: Read 72 rows from fc_stepar.
NOTE: Wrote stepar_h (12 rows, 2 columns).
NOTE: DATA elapsed:
  wall  0.00 seconds
  cpu   0.00 seconds
NOTE: DATA winters_h


NOTE: Read 72 rows from fc_winters.
NOTE: Wrote winters_h (12 rows, 2 columns).
NOTE: DATA elapsed:
  wall  0.00 seconds
  cpu   0.00 seconds
NOTE: DATA compare

NOTE: Stream 1 processed 12 rows, max BY-group size: 1 (O(1) memory verified)
NOTE: Stream 2 processed 12 rows, max BY-group size: 1 (O(1) memory verified)

NOTE: Wrote compare (12 rows, 4 columns).
NOTE: DATA elapsed:
  wall  0.00 seconds
  cpu   0.00 seconds
NOTE: PROC PRINT data=compare

NOTE: PROC PRINT completed: 12 observations printed, 4 variables


## Βήμα 5 — Πρόβλεψη κάθε κλάδου εργασιών ταυτόχρονα (επεξεργασία BY)

Οι πραγματικές εκτελέσεις αποθεμάτων καλύπτουν πολλά προϊόντα. Με τα δεδομένα ταξινομημένα κατά `line_of_business`, μια δήλωση `BY` κάνει την `PROC FORECAST` να προσαρμόσει ένα **ανεξάρτητο εποχιακό μοντέλο για κάθε ομάδα** σε μία μόνο κλήση. Εδώ συνθέτουμε ένα χαρτοφυλάκιο Αυτοκινήτου (υψηλότερος βασικός όγκος) και ένα χαρτοφυλάκιο Κατοικίας (χαμηλότερος βασικός όγκος) και προβλέπουμε και τα δύο έξι μήνες μπροστά. Το `OUTALL` γράφει το πλήρες σύνολο γραμμών — το ιστορικό `ACTUAL` και τον ορίζοντα `FORECAST` — μαζί με τις στήλες ορίων για κάθε ομάδα, και κρατάμε μόνο τις γραμμές `FORECAST` για τον παρακάτω πίνακα.

In [5]:
ΔΕΔΟΜΕΝΑ multi_book;
    CALL streaminit(20240531);
    LENGTH line_of_business $25;
    ΕΠΑΝΑΛΗΨΗ lob = 1 ΕΩΣ 2;
        ΕΑΝ lob = 1 ΤΟΤΕ line_of_business = 'Αυτοκίνητο';
        ΑΛΛΙΩΣ            line_of_business = 'Κατοικία';
        ΕΠΑΝΑΛΗΨΗ i = 0 ΕΩΣ 47;
            date = intnx('month', '01JAN2021'd, i);
            ΜΟΡΦΗ date monyy7.;
            mi   = mod(i, 12) + 1;
            BASE = (lob = 1) * 2000 + (lob = 2) * 1400;
            claim_count = round(BASE + 8 * i
                + 200 * sin((mi - 1) / 12 * 2 * constant('pi'))
                + 180 * (mi IN (12, 1, 2))
                + rand('normal', 0, 70));
            ΕΞΟΔΟΣ;
        ΤΕΛΟΣ;
    ΤΕΛΟΣ;
    ΚΡΑΤΗΣΗ line_of_business date claim_count;
ΕΚΤΕΛΕΣΗ;

ΔΙΑΔΙΚΑΣΙΑ ΤΑΞΙΝΟΜΗΣΗ ΔΕΔΟΜΕΝΑ=multi_book;
    ΚΑΤΑ line_of_business date;
ΕΚΤΕΛΕΣΗ;

ΔΙΑΔΙΚΑΣΙΑ forecast ΔΕΔΟΜΕΝΑ=multi_book
        out=fc_by
        METHOD=winters seasons=12 trend=2
        LEAD=6 ALPHA=0.05
        outall;
    ΚΑΤΑ line_of_business;
    id date interval=month;
    ΜΕΤΑΒΛΗΤΗ claim_count;
ΕΚΤΕΛΕΣΗ;

ΔΙΑΔΙΚΑΣΙΑ ΕΚΤΥΠΩΣΗ ΔΕΔΟΜΕΝΑ=fc_by(obs=12) ΕΤΙΚΕΤΑ;
    ΟΠΟΥ _type_ = 'FORECAST';
    ΕΤΙΚΕΤΑ line_of_business = 'Κλάδος Εργασιών' date = 'Ημερομηνία' _type_ = 'Τύπος'
          claim_count = 'Αξιώσεις' l95_claim_count = 'Κάτω 95%'
          u95_claim_count = 'Άνω 95%' residual_claim_count = 'Υπόλοιπο';
    TITLE 'Προβλέψεις 6 Μηνών ανά Κλάδο Εργασιών';
ΕΚΤΕΛΕΣΗ;

                                 Σύγκριση STEPAR έναντι Holt-Winters: Πρόβλεψη 12 Μηνών                                 


BY Group: line_of_business=Αυτοκίνητο

The FORECAST Procedure

Variables Forecasted: claim_count
Number of Observations: 48
Forecast Periods: 6


BY Group: line_of_business=Κατοικία

The FORECAST Procedure

Variables Forecasted: claim_count
Number of Observations: 48
Forecast Periods: 6



                                         Προβλέψεις 6 Μηνών ανά Κλάδο Εργασιών                                          

  Obs                Κλάδος Εργασιών            Ημερομηνία       Τύπος          Αξιώσεις      Κάτω 95%      Άνω 95%          Υπόλοιπο
    1  Αυτοκίνητο                                    23742  FORECAST         2524.596717   2359.095846  2690.097589                 .
    2  Αυτοκίνητο                                    23773  FORECAST         2604.418724   2370.365147    2838.4723                 .
    3  Αυτοκίνητο                                    23801  FO


NOTE: DATA multi_book


NOTE: Wrote multi_book (96 rows, 3 columns).
NOTE: DATA elapsed:
  wall  0.01 seconds
  cpu   0.01 seconds
NOTE: PROC SORT data=multi_book

NOTE: Unlicensed mode - input limited to 100 observations.
NOTE: Read 96 rows from multi_book.
NOTE: Wrote multi_book (96 rows, 3 columns).
NOTE: PROC SORT statement used.
NOTE: PROC FORECAST data=multi_book

NOTE: BY Group: line_of_business=Αυτοκίνητο
NOTE: Using Python for FORECAST estimation
NOTE: BY Group: line_of_business=Κατοικία
NOTE: Using Python for FORECAST estimation
NOTE: Output dataset created with 108 observations.
NOTE: PROC PRINT data=fc_by

NOTE: Unlicensed mode - input limited to 100 observations.
NOTE: PROC PRINT completed: 6 observations printed, 7 variables


## Ερμηνεία των αποτελεσμάτων

**Τι λένε τα μοντέλα στην ομάδα αποθεμάτων:**

- Το **STEPAR** αναπαράγει την ανοδική πορεία και τη βραχυπρόθεσμη ορμή, αλλά η 12μηνη πρόβλεψή του φθίνει ομαλά από περίπου 2.620 (βήμα 1) προς περίπου 1.980 στο μέσο του ορίζοντα, πριν επιστρέψει σε περίπου 2.140 — δεν αναπαράγει μια απότομη χειμερινή αιχμή, επειδή αντιμετωπίζει την εποχικότητα μόνο μέσω υστερήσεων αυτοπαλινδρόμησης. Είναι μια λογική γρήγορη βάση.
- Το **WINTERS** με `SEASONS=12` μεταφέρει τον ετήσιο ρυθμό απευθείας μέσω των πολλαπλασιαστικών εποχιακών συντελεστών του: η πρόβλεψή του είναι υψηλότερη στον πρώιμο ορίζοντα (περίπου 2.780 στο βήμα 1, περίπου 2.900 στο βήμα 2), υποχωρεί σε ένα χαμηλό κοντά στο βήμα 9 (περίπου 2.200), και ανεβαίνει ξανά μέχρι το βήμα 12 (περίπου 2.800). Έναντι της βάσης STEPAR βρίσκεται **150-660 αξιώσεις υψηλότερα** στο μεγαλύτερο μέρος του ορίζοντα (βλ. τη στήλη `Winters - STEPAR` στο Βήμα 4) — αυτό το κενό είναι το εποχιακό σήμα που το αυτοπαλινδρομικό μοντέλο αφήνει απ' έξω.
- Η **ζώνη εμπιστοσύνης 95%** (στήλες `L95_*`/`U95_*`, ελεγχόμενη από το `ALPHA=`) διευρύνεται καθώς επεκτείνεται ο ορίζοντας — για το μοντέλο WINTERS από ένα εύρος περίπου 410 αξιώσεων στο βήμα 1 σε περίπου 1.420 στο βήμα 12 — το ειλικρινές σήμα ότι οι εκτιμήσεις μακρινού ορίζοντα φέρουν περισσότερη αβεβαιότητα από τις βραχυπρόθεσμες. Οι αναλογιστές αποθεμάτων θα πρέπει να κρατούν κεφάλαιο έναντι του άνω ορίου, όχι μόνο της σημειακής πρόβλεψης.
- Η **επεξεργασία BY** παράγει τις προβλέψεις Αυτοκινήτου και Κατοικίας σε ένα πέρασμα, καθεμία με τη δική της εποχιακή προσαρμογή. Το χαρτοφυλάκιο Αυτοκινήτου προβλέπει στο εύρος περίπου 2.510-2.600 ενώ το χαρτοφυλάκιο Κατοικίας βρίσκεται κοντά στο 1.870-2.030, ώστε κάθε κλάδος να διατηρεί το δικό του επίπεδο και εποχιακό σχήμα — το μοτίβο που η ομάδα θα επέκτεινε σε κάθε προϊόν του χαρτοφυλακίου.

**Συμπέρασμα:** για μια σειρά πλήθους αξιώσεων με σαφή ετήσιο κύκλο, η `METHOD=WINTERS SEASONS=12` είναι η ενδεδειγμένη επιλογή· η βάση STEPAR είναι ένας χρήσιμος έλεγχος λογικής, και το `OUTFULL`/`OUTLIMIT` μαζί με μια βήμα-προς-βήμα σύγκριση μοντέλων επιτρέπουν στον αναλογιστή να προσδιορίσει το μπροστινό απόθεμα με τη ζώνη αβεβαιότητάς του.

> **Σημείωση πιστότητας μηχανής.** Αυτό το notebook καταγράφει δύο τρέχοντες περιορισμούς της PROC FORECAST του Jenner (δοκιμή κενού `400979`): το `ID` του ορίζοντα πρόβλεψης προάγεται κατά μία μονάδα ανά βήμα αντί κατά `INTERVAL=MONTH`, οπότε οι εκτυπωμένες ημερομηνίες πρόβλεψης δεν είναι οι προβλεπόμενοι ημερολογιακοί μήνες του 2025 (εξετάζουμε τον ορίζοντα ανά βήμα αντ' αυτού)· και τα `OUTRESID`/`OUTALL` δεν συμπληρώνουν ακόμη τις γραμμές ενός βήματος `_TYPE_='RESIDUAL'`, οπότε τα διαγνωστικά καταλοίπων αντικαθίστανται από μια άμεση σύγκριση δύο μοντέλων. Οι ίδιες οι σημειακές προβλέψεις και τα όρια εμπιστοσύνης παράγονται κανονικά και είναι αυτά στα οποία βασίζεται η παραπάνω αφήγηση.